# Video Frame Interpolation — IFRNet, minimal implementation

**From the README link:** Awesome-VFI — https://github.com/NK-CS-ZZL/Awesome-VFI
The list groups VFI methods as *flow-based* (RIFE, IFRNet, EMA-VFI, AMT, …), *kernel-based* (AdaCoF, …) and
hybrids. RIFE already has a notebook here (`rife.ipynb`), so this notebook covers another flow-based entry
from the list:

**IFRNet:** Kong et al., *IFRNet: Intermediate Feature Refine Network for Efficient Frame Interpolation*,
CVPR 2022 — https://arxiv.org/abs/2205.14620 · code: https://github.com/ltkong218/IFRNet

**How it works.** A shared pyramid encoder extracts features from both input frames `I0`, `I1`. Decoders run
coarse-to-fine, and at every level jointly refine (a) the two bidirectional flows from the target time `t` to
each input, and (b) an **intermediate feature** of the frame being synthesised. The finest decoder outputs flows,
a blending mask and an RGB residual:

`Î_t = M · warp(I0, F_t→0) + (1 − M) · warp(I1, F_t→1) + R`

Unlike RIFE there is no separate refinement net, and `t` can be any value in (0, 1).
The paper also adds a task-oriented flow distillation loss and a feature-space geometry consistency loss; this
version trains with a Charbonnier reconstruction loss only.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## Backward warping

In [2]:
def backward_warp(img, flow):
    """Sample `img` at positions displaced by `flow` (pixels, (B, 2, H, W) as dx, dy)."""
    B, _, H, W = img.shape
    ys, xs = torch.meshgrid(torch.arange(H, device=img.device), torch.arange(W, device=img.device), indexing="ij")
    gx = (xs[None] + flow[:, 0]) / (W - 1) * 2 - 1
    gy = (ys[None] + flow[:, 1]) / (H - 1) * 2 - 1
    return F.grid_sample(img, torch.stack([gx, gy], dim=-1), mode="bilinear", padding_mode="border", align_corners=True)

## Encoder and decoders

In [3]:
def conv(cin, cout, k=3, s=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, k // 2), nn.PReLU(cout))


class ResBlock(nn.Module):
    """IFRNet's residual block: a slice of the channels is refined separately at each conv."""
    def __init__(self, c, side=24):
        super().__init__()
        self.side = side
        self.c1, self.c2 = conv(c, c), conv(side, side)
        self.c3, self.c4 = conv(c, c), conv(side, side)
        self.c5 = nn.Conv2d(c, c, 3, 1, 1)
        self.act = nn.PReLU(c)

    def forward(self, x):
        out = self.c1(x)
        out = torch.cat([out[:, :-self.side], self.c2(out[:, -self.side:])], 1)
        out = self.c3(out)
        out = torch.cat([out[:, :-self.side], self.c4(out[:, -self.side:])], 1)
        return self.act(x + self.c5(out))


class Encoder(nn.Module):
    def __init__(self, ch=(32, 48, 72, 96)):
        super().__init__()
        cin = 3
        self.levels = nn.ModuleList()
        for c in ch:
            self.levels.append(nn.Sequential(conv(cin, c, 3, 2), conv(c, c)))
            cin = c

    def forward(self, x):  # returns features at 1/2, 1/4, 1/8, 1/16
        feats = []
        for lvl in self.levels:
            x = lvl(x)
            feats.append(x)
        return feats


class Decoder(nn.Module):
    """Takes [f0_warped, f1_warped, intermediate feature, flows (+ t)] and predicts 2x-upsampled updates."""
    def __init__(self, cin, cfeat, cout_extra):
        super().__init__()
        self.body = nn.Sequential(conv(cin, cfeat * 2), ResBlock(cfeat * 2),
                                  nn.ConvTranspose2d(cfeat * 2, 4 + cout_extra, 4, 2, 1))

    def forward(self, x):
        return self.body(x)


class IFRNet(nn.Module):
    def __init__(self, ch=(32, 48, 72, 96)):
        super().__init__()
        self.ch = ch
        self.encoder = Encoder(ch)
        # coarsest decoder: f0, f1 and the time embedding t
        self.dec4 = Decoder(ch[3] * 2 + 1, ch[3], ch[2])
        # middle decoders: warped f0, warped f1, intermediate feature, two flows (4 ch)
        self.dec3 = Decoder(ch[2] * 3 + 4, ch[2], ch[1])
        self.dec2 = Decoder(ch[1] * 3 + 4, ch[1], ch[0])
        # finest decoder: outputs flows + mask (1) + RGB residual (3)
        self.dec1 = Decoder(ch[0] * 3 + 4, ch[0], 1 + 3)

    def forward(self, img0, img1, t=0.5):
        B, _, H, W = img0.shape
        mean = torch.cat([img0, img1], 2).mean((1, 2, 3), keepdim=True)
        i0, i1 = img0 - mean, img1 - mean
        f0, f1 = self.encoder(i0), self.encoder(i1)
        if not torch.is_tensor(t):
            t = torch.full((B, 1, 1, 1), float(t), device=img0.device)
        t_map = t.view(B, 1, 1, 1).expand(B, 1, *f0[3].shape[-2:])

        out = self.dec4(torch.cat([f0[3], f1[3], t_map], 1))
        flow0, flow1, ft = out[:, 0:2], out[:, 2:4], out[:, 4:]

        for lvl, dec in ((2, self.dec3), (1, self.dec2), (0, self.dec1)):
            w0, w1 = backward_warp(f0[lvl], flow0), backward_warp(f1[lvl], flow1)
            out = dec(torch.cat([w0, w1, ft, flow0, flow1], 1))
            # flows are in pixels: upsample x2 and scale, then add the predicted residual flow
            flow0 = 2 * F.interpolate(flow0, scale_factor=2, mode="bilinear", align_corners=False) + out[:, 0:2]
            flow1 = 2 * F.interpolate(flow1, scale_factor=2, mode="bilinear", align_corners=False) + out[:, 2:4]
            ft = out[:, 4:]

        mask, residual = torch.sigmoid(ft[:, :1]), ft[:, 1:4]
        merged = mask * backward_warp(i0, flow0) + (1 - mask) * backward_warp(i1, flow1)
        return (merged + residual + mean).clamp(0, 1), (flow0, flow1, mask)

## Smoke test

Input height and width must be divisible by 16 (4 stride-2 levels).

In [4]:
model = IFRNet().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")  # the paper's IFRNet is ~5M

img0, img1 = torch.rand(2, 3, 128, 192, device=device), torch.rand(2, 3, 128, 192, device=device)
with torch.no_grad():
    out, (fl0, fl1, m) = model(img0, img1, t=0.5)
print("frame:", tuple(out.shape), " flow:", tuple(fl0.shape), " mask:", tuple(m.shape))

params: 3.45M
frame: (2, 3, 128, 192)  flow: (2, 2, 128, 192)  mask: (2, 1, 128, 192)


## Training on a toy task: moving squares

Triplets `(I0, It, I1)` of a square translating at constant velocity, with a random `t ∈ (0, 1)`.
A trained model should place the square at the right intermediate position; we compare against the naive
baseline of blending the two inputs.

In [5]:
def moving_squares(n, H=64, W=64, size=12):
    x0 = torch.randint(4, W - size - 20, (n,))
    y0 = torch.randint(4, H - size - 20, (n,))
    dx, dy = torch.randint(-12, 13, (n,)), torch.randint(-12, 13, (n,))
    x0, y0 = x0 + 12, y0 + 12  # keep everything in frame
    t = torch.rand(n)
    color = torch.rand(n, 3)
    def frame(tt):
        img = torch.full((n, 3, H, W), 0.1)
        xs = (x0 + tt * dx).round().long()
        ys = (y0 + tt * dy).round().long()
        for b in range(n):
            img[b, :, ys[b]:ys[b] + size, xs[b]:xs[b] + size] = color[b, :, None, None]
        return img
    return frame(torch.zeros(n)), frame(t), frame(torch.ones(n)), t


def charbonnier(a, b, eps=1e-3):
    return torch.sqrt((a - b) ** 2 + eps ** 2).mean()


opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for it in range(600):
    i0, it_gt, i1, t = (v.to(device) for v in moving_squares(32))
    pred, _ = model(i0, i1, t=t)
    loss = charbonnier(pred, it_gt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if it % 150 == 0 or it == 599:
        print(f"iter {it:4d}  loss {loss.item():.4f}")

iter    0  loss 0.0654


iter  150  loss 0.0153


iter  300  loss 0.0061


iter  450  loss 0.0034


iter  599  loss 0.0030


In [6]:
model.eval()
torch.manual_seed(123)
i0, it_gt, i1, t = (v.to(device) for v in moving_squares(64))
with torch.no_grad():
    pred, _ = model(i0, i1, t=t)
blend = (1 - t.view(-1, 1, 1, 1)) * i0 + t.view(-1, 1, 1, 1) * i1

def psnr(a, b):
    return (10 * torch.log10(1 / F.mse_loss(a, b, reduction="none").mean((1, 2, 3)))).mean().item()

print(f"PSNR  linear blend: {psnr(blend, it_gt):.2f} dB   IFRNet: {psnr(pred, it_gt):.2f} dB")

PSNR  linear blend: 29.22 dB   IFRNet: 36.05 dB
